In [0]:
# ============================================================
# bronze_offline_ingest.py
# Bronze-ingest op basis van geüploade offline dataset
# ============================================================

import pandas as pd
from datetime import datetime, timezone
import os

from config import (
    BRONZE_FILE,
    get_last_ingest_timestamp,
    update_last_ingest_timestamp,
    log_ingest,
    new_run_id,
)

# Pad naar het geüploade bestand
OFFLINE_FILE = "/Workspace/Users/<jouw-email>/offline_coingecko_prices.csv"

print(f"Laatste ingest timestamp: {get_last_ingest_timestamp()}")
RUN_ID = new_run_id()
print(f"RUN_ID: {RUN_ID}")

# ------------------------------------------------------------
# Offline dataset inladen
# ------------------------------------------------------------

df = pd.read_csv(OFFLINE_FILE, parse_dates=["price_timestamp"])

# ------------------------------------------------------------
# Incremental filtering
# ------------------------------------------------------------

last_ts_iso = get_last_ingest_timestamp()
last_ts = datetime.fromisoformat(last_ts_iso)

df_new = df[df["price_timestamp"] > last_ts]

if df_new.empty:
    print("Geen nieuwe data gevonden.")
else:
    # Voeg ingest metadata toe
    df_new["ingest_timestamp"] = datetime.now(timezone.utc)
    df_new["run_id"] = RUN_ID

    # Append naar Bronze Parquet
    if os.path.exists(BRONZE_FILE):
        df_existing = pd.read_parquet(BRONZE_FILE)
        df_all = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_all = df_new

    df_all.to_parquet(BRONZE_FILE, index=False)

    # Timestamp updaten
    max_ts = df_all["price_timestamp"].max()
    update_last_ingest_timestamp(max_ts.isoformat())

    print(f"{len(df_new)} nieuwe records toegevoegd.")
    print(f"Nieuwe ingest timestamp: {max_ts.isoformat()}")